In [ ]:
# installerar bibliotek
!pip install -q ultralytics opencv-python

In [ ]:
# laddar upp testbild
from google.colab import files
uploaded = files.upload()

In [ ]:
# laddar upp den tränade modellen
from google.colab import files
uploaded = files.upload()

In [ ]:
# testar modellen på en bild
from ultralytics import YOLO

model = YOLO("/content/best.pt")

results = model.predict(
    source="/content/IMG_4665.jpeg",
    conf=0.4,
    save=True
)

In [ ]:
!ls /content/runs/detect

In [ ]:
from PIL import Image
from IPython.display import display

display(Image.open("/content/runs/detect/predict/IMG_4665.jpg"))

In [ ]:
import math

model = YOLO("/content/best.pt")

# kör prediktion på bilden
results = model.predict(
    source="/content/IMG_4665.jpeg",
    conf=0.4,
    save=False
)

# hemtar resultatet från bilden
result = results[0]

# listar alla upteckta objekt
raw_detections = []

# går igenom alla bounding boxes som modellen hittat
for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    cls_id = int(box.cls[0].item())
    conf = float(box.conf[0].item())
    class_name = model.names[cls_id]

# räknar ut mitten på objektets box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

# sparar information om upptäckt objekt
    raw_detections.append({
        "class_id": cls_id,
        "class_name": class_name,
        "confidence": conf,
        "bbox": [x1, y1, x2, y2],
        "centroid": [cx, cy]
    })

# ta bort nästan identiska dubletter
filtered_detections = []

for det in raw_detections:
    keep = True
    for kept in filtered_detections:
        if det["class_id"] == kept["class_id"]:
            dx = det["centroid"][0] - kept["centroid"][0]
            dy = det["centroid"][1] - kept["centroid"][1]
            dist = math.sqrt(dx**2 + dy**2)

            # om samma klass och nästan samma plats -> behåll bara den med högst confidence
            if dist < 80:
                if det["confidence"] > kept["confidence"]:
                    filtered_detections.remove(kept)
                else:
                    keep = False
                break
    if keep:
        filtered_detections.append(det)

for d in filtered_detections:
    print(d)

In [ ]:
# räknar ut avståndet mellan alla uptäcjta objekt

for i in range(len(filtered_detections)):
    for j in range(i + 1, len(filtered_detections)):
        obj1 = filtered_detections[i]
        obj2 = filtered_detections[j]

        x1, y1 = obj1["centroid"]
        x2, y2 = obj2["centroid"]

        distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

        print(
            f"{obj1['class_name']} <-> {obj2['class_name']} : {distance:.2f} pixlar"
        )

In [ ]:
# gruppera objekt som ligger nära varandra

distance_threshold = 1800  # gränsen för hur nära objekt ska vra

# bygger adjacency lista för att koppla ihop objekt som ligger nära
n = len(filtered_detections)
adj = {i: [] for i in range(n)}

for i in range(n):
    for j in range(i + 1, n):
        x1, y1 = filtered_detections[i]["centroid"]
        x2, y2 = filtered_detections[j]["centroid"]

        distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

        if distance < distance_threshold:
            adj[i].append(j)
            adj[j].append(i)

# hitta grupper med DFS
visited = set()
groups = []

def dfs(node, group):
    visited.add(node)
    group.append(node)
    for neighbor in adj[node]:
        if neighbor not in visited:
            dfs(neighbor, group)

for i in range(n):
    if i not in visited:
        group = []
        dfs(i, group)
        groups.append(group)

# skriv ut grupperna
for idx, group in enumerate(groups):
    print(f"Grupp {idx+1}:")
    for obj_index in group:
        obj = filtered_detections[obj_index]
        print("  ", obj["class_name"], obj["centroid"])

In [ ]:
# bestämmer status för varje grupp: ensam, sorterad eller osorterad
group_statuses = []

for idx, group in enumerate(groups):
    class_names = [filtered_detections[i]["class_name"] for i in group]
    unique_classes = set(class_names)

    if len(group) == 1:
        status = "alone"
    elif len(unique_classes) == 1:
        status = "sorted"
    else:
        status = "unsorted"

    group_statuses.append({
        "group_id": idx,
        "members": group,
        "classes": class_names,
        "status": status
    })

for g in group_statuses:
    print(f"Grupp {g['group_id']+1}: {g['status']} -> {g['classes']}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

# läs originalbilden
image_path = "/content/IMG_4665.jpeg"
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# pluralnamn för sorterade grupper
plural_names = {
    "Dryckeskartong": "KARTONGER",
    "Konservburk": "KONSERVBURKAR",
    "Pantburk": "PANTBURKAR"
}

for group_info in group_statuses:
    members = group_info["members"]
    status = group_info["status"]

    # välj färg
    if status == "sorted":
        color = (0, 255, 0)      # grön
    elif status == "unsorted":
        color = (255, 0, 0)      # röd
    else:
        color = (255, 255, 0)    # gul

    # rita varje box i gruppen
    for obj_index in members:
        det = filtered_detections[obj_index]
        x1, y1, x2, y2 = map(int, det["bbox"])

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 8)

        # individuell text för unsorted och alone
        if status == "unsorted":
            cv2.putText(
                img,
                det["class_name"],
                (x1, max(y1 - 20, 40)),
                cv2.FONT_HERSHEY_SIMPLEX,
                2.0,
                color,
                5,
                cv2.LINE_AA
            )

        elif status == "alone":
            cv2.putText(
                img,
                f"{det['class_name']} (ensam)",
                (x1, max(y1 - 20, 40)),
                cv2.FONT_HERSHEY_SIMPLEX,
                2.0,
                color,
                5,
                cv2.LINE_AA
            )

    # gemensam grupptext för sorted
    if status == "sorted":
        first_class = filtered_detections[members[0]]["class_name"]
        label = plural_names.get(first_class, first_class.upper())

        min_x = min(int(filtered_detections[i]["bbox"][0]) for i in members)
        min_y = min(int(filtered_detections[i]["bbox"][1]) for i in members)

        cv2.putText(
            img,
            label,
            (min_x, max(min_y - 30, 50)),
            cv2.FONT_HERSHEY_SIMPLEX,
            2.2,
            color,
            6,
            cv2.LINE_AA
        )

plt.figure(figsize=(12, 12))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
# laddar upp video
from google.colab import files
uploaded = files.upload()

In [ ]:
# testar om videon går att öppna
import cv2

cap = cv2.VideoCapture("/content/IMG_4685.MOV")

print("Opened:", cap.isOpened())

ret, frame = cap.read()
print("Frame loaded:", ret)

cap.release()

In [ ]:
import cv2
import math
from ultralytics import YOLO
from collections import defaultdict, deque

model = YOLO("/content/best.pt")

input_video = "/content/IMG_4685.MOV"
output_video = "/content/output_demo_improved2.mp4"

cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

plural_names = {
    "Dryckeskartong": "KARTONGER",
    "Konservburk": "KONSERVBURKAR",
    "Pantburk": "PANTBURKAR"
}

# historik för att stabilisera status mellan frames
group_status_history = defaultdict(lambda: deque(maxlen=3))

def stable_status(group_key, new_status):
    history = group_status_history[group_key]
    history.append(new_status)
    if len(history) == 3 and len(set(history)) == 1:
        return history[-1]
    return history[-1]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # tracking ger stabilare resultat än predict
    results = model.track(frame, conf=0.45, persist=True, verbose=False)
    result = results[0]

    raw_detections = []

    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            class_name = model.names[cls_id]

            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2

            raw_detections.append({
                "class_id": cls_id,
                "class_name": class_name,
                "confidence": conf,
                "bbox": [x1, y1, x2, y2],
                "centroid": [cx, cy]
            })

    # ta bort nästan identiska dubletter
    filtered_detections = []
    for det in raw_detections:
        keep = True
        for kept in filtered_detections:
            if det["class_id"] == kept["class_id"]:
                dx = det["centroid"][0] - kept["centroid"][0]
                dy = det["centroid"][1] - kept["centroid"][1]
                dist = math.sqrt(dx**2 + dy**2)

                if dist < 60:
                    if det["confidence"] > kept["confidence"]:
                        filtered_detections.remove(kept)
                    else:
                        keep = False
                    break
        if keep:
            filtered_detections.append(det)

    # bygg grupper
    n = len(filtered_detections)
    adj = {i: [] for i in range(n)}

    for i in range(n):
        for j in range(i + 1, n):
            x1, y1 = filtered_detections[i]["centroid"]
            x2, y2 = filtered_detections[j]["centroid"]

            dx = abs(x2 - x1)
            dy = abs(y2 - y1)
            distance = math.sqrt(dx**2 + dy**2)

            bx1_i, by1_i, bx2_i, by2_i = filtered_detections[i]["bbox"]
            w_i = bx2_i - bx1_i
            h_i = by2_i - by1_i

            bx1_j, by1_j, bx2_j, by2_j = filtered_detections[j]["bbox"]
            w_j = bx2_j - bx1_j
            h_j = by2_j - by1_j

            diag_i = math.sqrt(w_i**2 + h_i**2)
            diag_j = math.sqrt(w_j**2 + h_j**2)

            dynamic_threshold = 0.9 * max(diag_i, diag_j)

            # kräver att de både är nära i totalavstånd och inte för långt isär i sidled
            if distance < dynamic_threshold and dx < dynamic_threshold:
                adj[i].append(j)
                adj[j].append(i)

    visited = set()
    groups = []

    def dfs(node, group):
        visited.add(node)
        group.append(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                dfs(neighbor, group)

    for i in range(n):
        if i not in visited:
            group = []
            dfs(i, group)
            groups.append(group)

    # klassificera grupper
    group_statuses = []

    for idx, group in enumerate(groups):
        class_names = [filtered_detections[i]["class_name"] for i in group]
        unique_classes = set(class_names)

        if len(group) == 1:
            status = "alone"
        elif len(unique_classes) == 1:
            status = "sorted"
        else:
            status = "unsorted"

        # skapa enkel gruppnyckel baserad på medelcentroid
        avg_x = int(sum(filtered_detections[i]["centroid"][0] for i in group) / len(group))
        avg_y = int(sum(filtered_detections[i]["centroid"][1] for i in group) / len(group))
        group_key = (avg_x // 80, avg_y // 80)

        status = stable_status(group_key, status)

        group_statuses.append({
            "group_id": idx,
            "members": group,
            "classes": class_names,
            "status": status
        })

    # rita på frame
    for group_info in group_statuses:
        members = group_info["members"]
        status = group_info["status"]

        if status == "sorted":
            color = (0, 255, 0)      # grön
        elif status == "unsorted":
            color = (0, 0, 255)      # röd
        else:
            color = (0, 255, 255)    # gul

        for obj_index in members:
            det = filtered_detections[obj_index]
            x1, y1, x2, y2 = map(int, det["bbox"])

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

            if status == "unsorted":
                cv2.putText(
                    frame,
                    det["class_name"],
                    (x1, max(y1 - 12, 30)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2,
                    cv2.LINE_AA
                )

            elif status == "alone":
                cv2.putText(
                    frame,
                    f"{det['class_name']} (ensam)",
                    (x1, max(y1 - 12, 30)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2,
                    cv2.LINE_AA
                )

        if status == "sorted" and len(members) > 0:
            first_class = filtered_detections[members[0]]["class_name"]
            label = plural_names.get(first_class, first_class.upper())

            min_x = min(int(filtered_detections[i]["bbox"][0]) for i in members)
            min_y = min(int(filtered_detections[i]["bbox"][1]) for i in members)

            cv2.putText(
                frame,
                label,
                (min_x, max(min_y - 15, 30)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                color,
                3,
                cv2.LINE_AA
            )

    out.write(frame)

cap.release()
out.release()

print("Klar:", output_video)

In [ ]:
from google.colab import files
files.download("/content/output_demo_improved2.mp4")